In [1]:
import numpy as np
import h5py
from scipy.io import savemat
# from utils import GenerateSyntheticNoise, DecodeEventBasedRawData

In [2]:
# variables
fileDirectory = 'C:\\Users\\Admin\\UniBE\\data\\220525_530_19_1b_DRC470_02\\'
fileName = '220525_530_19_1b_DRC470_02.brw'
wellID = 'Well_A1'

# open the BRW file
file = h5py.File(fileDirectory + fileName, 'r')

In [3]:
with file as f:
    for key in f:
        print(f"Group: {key}")
        for subkey in f[key]:
            print(f"  Subgroup: {subkey}")

Group: ExperimentSettings
  Subgroup: b'{\r\n  "TimelineEvents": [],\r\n  "ValueConverter": {\r\n    "MinAnalogValue": -8000.0,\r\n    "MaxAnalogValue": 8000.0,\r\n    "MaxDigitalValue": 4095,\r\n    "MinDigitalValue": 0,\r\n    "ScaleFactor": 1.0,\r\n    "JsonVersion": 0\r\n  },\r\n  "TimeConverter": {\r\n    "FrameRate": 10103.0751953125\r\n  },\r\n  "ExperimentType": "Standard",\r\n  "ExperimentSettings": {\r\n    "Description": null,\r\n    "CalibrationCh": {\r\n      "Well": 1,\r\n      "Row": 1,\r\n      "Col": 1\r\n    },\r\n    "JsonVersion": 0\r\n  },\r\n  "SourceFilterSettings": {\r\n    "HighPassFilterCutOffFrequency": 100.0,\r\n    "HighPassFilterOrder": 2,\r\n    "JsonVersion": 0\r\n  },\r\n  "BiologicalModel": "Retina",\r\n  "TimelineIntervals": [],\r\n  "DataSettings": {\r\n    "Noise": {\r\n      "$type": "NoiseInfoDetectionSettings",\r\n      "DiscardOutliers": true,\r\n      "JsonVersion": 0\r\n    },\r\n    "EventsBasedRawRanges": {\r\n      "$type": "NoiseBlankingCo

In [3]:
# collect experiment information
minDigitalValue = file.attrs['MinDigitalValue'] 
maxDigitalValue = file.attrs['MaxDigitalValue'] 
minAnalogValue = file.attrs['MinAnalogValue'] 
maxAnalogValue = file.attrs['MaxAnalogValue'] 
samplingRate = file.attrs['SamplingRate']
chIdxs = np.array(file[wellID + '/StoredChIdxs'])

# data 
raw = np.array(file[wellID + '/EventsBasedSparseRaw'])
TOC = np.array(file[wellID + '/EventsBasedSparseRawTOC'])

In [4]:
# initialize an empty (fill with zeros) data collection 
data = {} 
for chIdx in chIdxs: 
    data[chIdx] = []

In [5]:
def DecodeEventBasedRawData(file, data, wellID):  
    # decode all data for the given well ID and time interval 
    binaryData = file[wellID + '/EventsBasedSparseRaw'][:] 
    binaryDataLength = len(binaryData)
 
    pos = 0 
    while pos < binaryDataLength: 
        chIdx = int.from_bytes(binaryData[pos:pos + 4], byteorder='little', signed=True) 
        print("channel", chIdx)
        pos += 4 
 
        chDataLength = int.from_bytes(binaryData[pos:pos + 4], byteorder='little', signed=True) 
        pos += 4 
 
        chDataPos = pos 
        while pos < chDataPos + chDataLength:  # loop till the end of the byte=pos+length of byte
            # Get start and end frames for this chunk of recorded date
            fromInclusive = int.from_bytes(binaryData[pos:pos + 8], byteorder='little', signed=True) # Start frame
            pos += 8 
 
            toExclusive = int.from_bytes(binaryData[pos:pos + 8], byteorder='little', signed=True)  # End frame 
            pos += 8 

            print('length of detected event', toExclusive-fromInclusive)
            # print(fromInclusive, toExclusive)
 
            rangeDataPos = pos 
            lst = []
            # Loop within the actual spike event. Each iteration corresponds to one sample point recorded by device at frame j
            for j in range(fromInclusive, toExclusive):  # Loop within the actual spike event
                lst.append(int.from_bytes(binaryData[rangeDataPos:rangeDataPos + 2], byteorder='little', signed=True)) # Each sample (spike amplitude) is 2 bytes.
                # print('data', int.from_bytes(binaryData[rangeDataPos:rangeDataPos + 2], byteorder='little', signed=True))
                rangeDataPos += 2 # Move to the next point (forward by 2 bytes)
            pos += (toExclusive - fromInclusive) * 2  # skip ahead by 2 bytes per frame
            # arr = np.array(lst, dtype=np.int32)
            # print("inner array dtype:", arr.dtype) 
            data[chIdx].append(lst)
        if chIdx == 4:
            break   

In [6]:
DecodeEventBasedRawData(file, data, wellID)

channel 0
length of detected event 2526
channel 1
length of detected event 2526
channel 8
length of detected event 106
channel 10
length of detected event 106
length of detected event 137
channel 13
length of detected event 129
channel 16
length of detected event 106
length of detected event 106
length of detected event 173
length of detected event 106
length of detected event 117
length of detected event 106
channel 21
length of detected event 106
length of detected event 106
length of detected event 112
channel 23
length of detected event 106
channel 24
length of detected event 106
length of detected event 107
channel 25
length of detected event 106
channel 26
length of detected event 126
channel 27
length of detected event 106
length of detected event 111
channel 28
length of detected event 157
length of detected event 107
length of detected event 106
length of detected event 106
length of detected event 106
length of detected event 106
channel 30
length of detected event 278
length

In [7]:
new = {}
for key in data.keys():
    str_key =  f'ch_{key}'
    if data[key]:
        new[str_key] = np.zeros(len(data[key]), dtype=object)
        for i in range(len(data[key])):
            new[str_key][i] = np.array(data[key][i], dtype=int)

In [8]:
new

{'ch_0': array([array([2048, 2048, 2048, ..., 2048, 2048, 2048], shape=(2526,)),
        array([2048, 2048, 2048, ..., 2048, 2048, 2048], shape=(2526,))],
       dtype=object),
 'ch_1': array([array([2049, 2049, 2049, ..., 2049, 2050, 2054], shape=(2526,)),
        array([2058, 2059, 2059, ..., 2052, 2051, 2050], shape=(2526,))],
       dtype=object),
 'ch_3': array([array([2049, 2045, 2048, 2052, 2052, 2054, 2052, 2047, 2050, 2045, 2047,
               2051, 2053, 2040, 2031, 2032, 2029, 2028, 2037, 2034, 2038, 2038,
               2045, 2051, 2050, 2051, 2047, 2047, 2047, 2048, 2051, 2061, 2059,
               2051, 2056, 2056, 2054, 2052, 2057, 2056, 2054, 2054, 2051, 2051,
               2051, 2048, 2052, 2050, 2051, 2060, 2056, 2059, 2050, 2056, 2054,
               2042, 2040, 2046, 2043, 2046, 2057, 2055, 2046, 2044, 2052, 2060,
               2054, 2049, 2052, 2055, 2054, 2045, 2045, 2050, 2051, 2051, 2053,
               2045, 2044, 2041, 2041, 2046, 2043, 2043, 2051, 2048, 20

In [9]:
savemat("output.mat", new)

In [10]:
from scipy.io import loadmat
data = loadmat('output.mat')

In [11]:
data

{'__header__': b'MATLAB 5.0 MAT-file Platform: nt, Created on: Mon Jun 30 14:56:10 2025',
 '__version__': '1.0',
 '__globals__': [],
 'ch_0': array([[array([[2048, 2048, 2048, ..., 2048, 2048, 2048]], shape=(1, 2526)),
         array([[2048, 2048, 2048, ..., 2048, 2048, 2048]], shape=(1, 2526))]],
       dtype=object),
 'ch_1': array([[array([[2049, 2049, 2049, ..., 2049, 2050, 2054]], shape=(1, 2526)),
         array([[2058, 2059, 2059, ..., 2052, 2051, 2050]], shape=(1, 2526))]],
       dtype=object),
 'ch_3': array([[array([[2049, 2045, 2048, 2052, 2052, 2054, 2052, 2047, 2050, 2045, 2047,
                 2051, 2053, 2040, 2031, 2032, 2029, 2028, 2037, 2034, 2038, 2038,
                 2045, 2051, 2050, 2051, 2047, 2047, 2047, 2048, 2051, 2061, 2059,
                 2051, 2056, 2056, 2054, 2052, 2057, 2056, 2054, 2054, 2051, 2051,
                 2051, 2048, 2052, 2050, 2051, 2060, 2056, 2059, 2050, 2056, 2054,
                 2042, 2040, 2046, 2043, 2046, 2057, 2055, 2046, 204